In [1]:
##### Converts raw rasters on crop and livestock production into final varaiables needed 
# pixel and subnational (vector) scale
# variables: 
    # % of production by crop/livestock grouping

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from collections import defaultdict
from rasterio.features import geometry_mask

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# total production 
total_production = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# Save paths
vectors = f"{cd}/Data/Clean/Predictors/Vectors/commodity_mix.csv"

In [3]:
##### Define groupings

crop_groups = {
    # cereals
    'WHEA': 'cereals',
    'RICE': 'cereals',
    'BARL': 'cereals',
    'MAIZ': 'cereals',
    'OCER': 'cereals',
    'AGG_MILLET': 'cereals',
    'SORG': 'cereals',
    # fibres
    'COTT': 'fibres',
    'OFIB': 'fibres',
    # fruits
    'BANA': 'fruits',
    'PLNT': 'fruits',
    'CITR': 'fruits',
    'TEMF': 'fruits',
    'TROF': 'fruits',
    # oilcrops
    'SOYB': 'oilcrops',
    'GROU': 'oilcrops',
    'CNUT': 'oilcrops',
    'OILP': 'oilcrops',
    'OOIL': 'oilcrops',
    'SUNF': 'oilcrops',
    'RAPE': 'oilcrops',
    'SESA': 'oilcrops',
    # pulses
    'BEAN': 'pulses',
    'OPUL': 'pulses',
    'CHIC': 'pulses',
    'COWP': 'pulses',
    'PIGE': 'pulses',
    'LENT': 'pulses',
    # roots & tubers
    'POTA': 'roots_tubers',
    'SWPO': 'roots_tubers',
    'CASS': 'roots_tubers',
    'ORTS': 'roots_tubers',
    'YAMS': 'roots_tubers',
    # rest of crops
    'REST': 'rest_of_crops',
    'AGG_COFFEE': 'rest_of_crops',
    'COCO': 'rest_of_crops',
    'TEAS': 'rest_of_crops',
    'TOBA': 'rest_of_crops',
    # sugar crops
    'SUGC': 'sugar_crops',
    'SUGB': 'sugar_crops',
    # vegetables
    'VEGE': 'vegetables',
    'TOMA': 'vegetables',
    'ONIO': 'vegetables',
    # rubber
    'RUBB': 'rubber',
}

livestock_groups = {
    # ruminants
    'cattle': 'ruminants',
    'buffalo': 'ruminants',
    'sheep': 'ruminants',
    'goats': 'ruminants',
    # monogastrics
    'horses': 'monogastrics',
    'pigs': 'monogastrics',
    # poultry
    'ducks': 'poultry',
    'chicken': 'poultry',
    # other
    'all': 'other'
}

In [4]:
#### Run resampling for pixel scale 

### PREP 

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    dst_transform = ref.transform
    dst_crs       = ref.crs
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask: True where reference raster HAS data
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

# load total production 
with rasterio.open(total_production) as src:
    total = src.read(1).astype(np.float64)
    total[total == -9999] = np.nan

# reproject countries to match reference raster CRS if needed
if countries.crs != dst_crs:
    countries = countries.to_crs(dst_crs)
countries = countries.reset_index(drop=True)

# invert groups to get {group: [crop1, crop2, ...]}
group_crops = defaultdict(list)
for crop, group in crop_groups.items():
    group_crops[group].append(crop)

group_livestock = defaultdict(list)
for animal, group in livestock_groups.items():
    group_livestock[group].append(animal)


### GAP-FILL HELPER
def fill_with_country_means(arr, label):
    """
    Given a float32 array (NaN = nodata), fill pixels that are:
      - valid in the reference raster, AND
      - NaN in arr
    using the country-level mean of arr. Returns the filled array.
    """
    # Step 1: blank out pixels outside the reference mask
    arr[~ref_valid] = np.nan

    needs_fill = ref_valid & np.isnan(arr)
    print(f"  [{label}] Pixels needing fill: {needs_fill.sum()}")

    if not needs_fill.any():
        return arr

    # country-level means from the partial array
    country_stats = rasterstats.zonal_stats(
        countries,
        arr,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )

    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }

    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue

        country_mask = geometry_mask(
            [row.geometry],
            transform=dst_transform,
            invert=True,
            out_shape=dst_shape,
        )

        to_fill = needs_fill & country_mask
        fill_array[to_fill] = mean_val

    arr = np.where(needs_fill, fill_array, arr)

    still_missing = ref_valid & np.isnan(arr)
    if still_missing.any():
        print(f"  [{label}] Warning: {still_missing.sum()} pixels still unfilled "
              f"(country had no valid data). These will be saved as NoData.")

    return arr


### CALCULATE CROPS
# for each group: sum crops in group then divide by total
for group, crops in group_crops.items():

    group_sum = np.zeros(dst_shape, dtype=np.float64)

    for crop in crops:
        path = f"{cd}/Data/Clean/Production/Crop_value/{crop}_crop_value_USD_2020.tif"
        try:
            with rasterio.open(path) as src:
                data = src.read(1).astype(np.float64)
                data[data == -9999] = np.nan
                if src.nodata is not None:
                    data[data == src.nodata] = np.nan
                group_sum = np.where(np.isnan(data), group_sum, group_sum + np.nan_to_num(data, nan=0))
        except Exception as e:
            print(f"WARNING: could not load {crop} — {e}")

    # divide by total to get share
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total > 0) & ~np.isnan(total),
            group_sum / total,
            np.nan
        ).astype(np.float32)

    share = np.clip(share, 0, 1).astype(np.float32)

    # gap-fill
    share = fill_with_country_means(share, group)

    # save
    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = share.copy()
    out_arr[np.isnan(out_arr)] = -9999

    out_path = f"{cd}/Data/Clean/Predictors/Rasters/{group}_share_production_USD.tif"
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved {group} ({len(crops)} crops) → {out_path}")


### CALCULATE LIVESTOCK
# for each group: sum livestock in group then divide by total
for group, animals in group_livestock.items():

    group_sum = np.zeros(dst_shape, dtype=np.float64)

    for animal in animals:
        path = f"{cd}/Data/Clean/Production/Livestock_value/{animal}_production_USD_2020.tif"
        try:
            with rasterio.open(path) as src:
                data = src.read(1).astype(np.float64)
                data[data == -9999] = np.nan
                if src.nodata is not None:
                    data[data == src.nodata] = np.nan
                group_sum = np.where(np.isnan(data), group_sum, group_sum + np.nan_to_num(data, nan=0))
        except Exception as e:
            print(f"WARNING: could not load {animal} — {e}")

    # share of total production
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total > 0) & ~np.isnan(total),
            group_sum / total,
            np.nan
        ).astype(np.float32)

    share = np.clip(share, 0, 1).astype(np.float32)

    # gap-fill
    share = fill_with_country_means(share, group)

    # save
    out_meta = dst_meta.copy()
    out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

    out_arr = share.copy()
    out_arr[np.isnan(out_arr)] = -9999

    out_path = f"{cd}/Data/Clean/Predictors/Rasters/{group}_share_production_USD.tif"
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_arr, 1)

    print(f"Saved {group} ({len(animals)} animals) → {out_path}")

  [cereals] Pixels needing fill: 152560
  [cereals] Warning: 1 pixels still unfilled (country had no valid data). These will be saved as NoData.
Saved cereals (7 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/cereals_share_production_USD.tif
  [fibres] Pixels needing fill: 152560
  [fibres] Warning: 1 pixels still unfilled (country had no valid data). These will be saved as NoData.
Saved fibres (2 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/fibres_share_production_USD.tif
  [fruits] Pixels needing fill: 152560
  [fruits] Warning: 1 pixels still unfilled (country had no valid data). These will be saved as NoData.
Saved fruits (5 crops) → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/fruits_share_production_USD.tif
  [oilcrops] Pixels needing fill: 152560
  [oilcrops] Warning: 1 pixels still unfilled (country had no valid data). These will be saved as NoData.


In [5]:
#### Run resampling for vector scale 
### TAKES 1 HOUR TO RUN ###

### PREP 
# reproject geo
gdf_proj  = total_geo.to_crs("EPSG:4326")

# prep results 
result = total_geo[["PROJ_ID"]].copy()


### CALCULATE
# step 1: sum total production per vector
zonal_total = rasterstats.zonal_stats(gdf_proj, total_production, stats="sum", nodata=-9999)
total_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_total])

# get crop shares
for group, crops in group_crops.items():

    group_sums = np.zeros(len(gdf_proj), dtype=np.float64)

    for crop in crops:
        path = f"{cd}/Data/Clean/Production/Crop_value/{crop}_crop_value_USD_2020.tif"
        try:
            zonal = rasterstats.zonal_stats(gdf_proj, path, stats="sum", nodata=-9999)
            vals  = np.array([z["sum"] if z["sum"] is not None else 0.0 for z in zonal])
            group_sums += np.nan_to_num(vals, nan=0)
        except Exception as e:
            print(f"WARNING: could not load {crop} — {e}")

    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total_sums > 0) & ~np.isnan(total_sums),
            group_sums / total_sums,
            np.nan
        )

    result[f"{group}_share_production_USD"] = np.clip(share, 0, 1)
    print(f"Done: {group}")

# get livestock shares
for group, animals in group_livestock.items():

    group_sums = np.zeros(len(gdf_proj), dtype=np.float64)

    for animal in animals:
        path = f"/{cd}/Data/Clean/Production/Livestock_value/{animal}_production_USD_2020.tif"
        try:
            zonal = rasterstats.zonal_stats(gdf_proj, path, stats="sum", nodata=-9999)
            vals  = np.array([z["sum"] if z["sum"] is not None else 0.0 for z in zonal])
            group_sums += np.nan_to_num(vals, nan=0)
        except Exception as e:
            print(f"WARNING: could not load {animal} — {e}")

    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(
            (total_sums > 0) & ~np.isnan(total_sums),
            group_sums / total_sums,
            np.nan
        )

    result[f"{group}_share_production_USD"] = np.clip(share, 0, 1)
    print(f"Done: {group}")

result.to_csv(vectors, index=False)


Done: cereals
Done: fibres
Done: fruits
Done: oilcrops
Done: pulses
Done: roots_tubers
Done: rest_of_crops
Done: sugar_crops
Done: vegetables
Done: rubber
Done: ruminants
Done: monogastrics
Done: poultry
Done: other
